# DeepDiscover with CAMELS-US data

This notebook uses the daily CAMELS-US forcing and streamflow records for basin `01013500`. Obtain these files from the original CAMELS-US data provider; they are not included in DeepDiscover. Place them in `data/camels/01013500/` as described in the [CAMELS-US instructions](../docs/camels.md).

The example uses the two-DU serial configuration from the Multiple-DU study. Precipitation and air temperature drive the snow module. Its rainfall and snowmelt outputs, together with PET, are passed to the DiscoverUnits.

In [ ]:
from pathlib import Path
from deepdiscover import (
    DeepDiscover, load_camels_dataset, CRCNet, DiscoverUnit,
    MultiDUModel, DemandLimitedSink, DifferentiableDegreeDaySnow,
)

basin = '01013500'
forcing = Path('data/camels') / basin / f'{basin}_lump_cida_forcing_leap.txt'
flow = Path('data/camels') / basin / f'{basin}_streamflow_qc.txt'
if not forcing.exists() or not flow.exists():
    raise FileNotFoundError('Download the CAMELS-US forcing and streamflow files first. See docs/camels.md')

data = load_camels_dataset(
    '.', basin, n_du=2, start='1980-10-01', end='2010-09-30'
)
print('Water:', data.water.shape, 'Drivers (P, PET, T):', data.drivers.shape)

## Model configuration

The first DU has a direct discharge process and a transfer to the second DU. The second DU releases water to the outlet. A demand-limited evapotranspiration loss acts on the first DU. The three process networks and their input scales follow the Multiple-DU CAMELS configuration.

The data contain informational drivers `(P, PET, T)`. The `driver_channels` setting supplies `(rain, snowmelt, PET)` to the hydrological networks after the snow calculation. The temperature is used by the snow module.

In [ ]:
import keras

keras.utils.set_random_seed(42)

rain_ref = 10.0
melt_ref = 10.0
storage_ref = 100.0
flow_ref = 10.0
hidden_units = 16

et_efficiency = CRCNet(
    [1], [storage_ref], hidden_units=hidden_units, output='sigmoid'
)
first_du = DiscoverUnit(
    process_nets=[
        CRCNet([1, 1, 1], [rain_ref, melt_ref, storage_ref], hidden_units=hidden_units),
        CRCNet([1, 1, 1], [rain_ref, melt_ref, storage_ref], hidden_units=hidden_units),
    ],
    context_indices=[(0, 1), (0, 1)],
    exchanges=[DemandLimitedSink(et_efficiency, demand_index=2)],
)
second_du = DiscoverUnit(
    process_nets=[CRCNet([1, 1], [storage_ref, flow_ref], hidden_units=hidden_units)],
    context_indices=[()], upstream_count=1,
)

model = DeepDiscover(
    MultiDUModel(
        [first_du, second_du],
        destinations=[['outlet', 1], ['outlet']],
    ),
    observations={'streamflow': 'outlet'},
    snow=DifferentiableDegreeDaySnow(),
    snow_unit=0,
    temperature_index=2,
    driver_channels=('rain', 'snowmelt', 1),  # PET occupies raw column 1
)
print(model.describe())

## Chronological periods and training windows

The scored periods match the Multiple-DU CAMELS study. Training uses 19 independent windows, each with 365 warm-up days and 365 scored days. Hence the tensors have shape `(B, T, n)` with `B=19` and `T=730`. The warm-up observations are masked out of the objective. Validation and test each use a single continuous period preceded by 365 warm-up days.

Full mode uses all 19 training windows in one gradient update. The `benchmark` mode measures the cost of three such updates. The `check` mode is only a short installation example.

In [ ]:
import os

mode = os.environ.get('DEEPDISCOVER_CAMELS_MODE', 'full').lower()
if mode not in ('check', 'benchmark', 'full'):
    raise ValueError('DEEPDISCOVER_CAMELS_MODE must be check, benchmark or full')

train_end = 365 + 6940
validation_end = train_end + 1826
assert len(data.water) == validation_end + 1826

if mode == 'check':
    train = data.make_windows(
        warmup=45, score_length=30, stride=60,
        first_score=365, last_score=365 + 365,
    )
    validation = data.continuous_batch(train_end, train_end + 120, warmup=45)
    test = data.continuous_batch(validation_end, validation_end + 120, warmup=45)
    n_epochs = 2
else:
    train = data.make_windows(
        warmup=365, score_length=365, stride=365,
        first_score=365, last_score=train_end,
    )
    validation = data.continuous_batch(train_end, validation_end, warmup=365)
    test = data.continuous_batch(validation_end, len(data.water), warmup=365)
    n_epochs = 3 if mode == 'benchmark' else 400

print('Mode:', mode)
print('Train water:', train.water.shape, 'Drivers:', train.drivers.shape)
print('Train observed:', train.targets['streamflow'].shape)
print('Validation:', validation.water.shape, 'Test:', test.water.shape)
print('Scored train values:', int(train.masks['streamflow'].sum()))
if mode != 'check':
    print('Unscored final training days:', 6940 - int(train.masks['streamflow'].sum()))
print('Epochs:', n_epochs, flush=True)

## Training and evaluation

Full mode follows the Multiple-DU training-window configuration, learning rate and validation frequency. The upper limit is 400 epochs, with early stopping if validation does not improve for 20 successive checks. A check is performed every five epochs. The lowest validation loss is used to restore the final weights.

The `benchmark` mode uses the same complete training windows and parallel batch, but only three epochs. It can be used to examine execution time before starting full calibration. Its hydrological scores have no interpretive value.

In [ ]:
history = model.fit(
    train, validation,
    epochs=n_epochs,
    batch_size=None,  # all B windows processed together
    learning_rate=0.01,
    validation_every=5 if mode != 'check' else 1,
    patience=20 if mode == 'full' else None,
    normalize=False,
    shuffle=False,
    verbose=2,
)
for item in history:
    if item['epoch'] <= 3 or 'validation_loss' in item:
        print('Timing:', item, flush=True)
if mode == 'full':
    import json

    best = min((record for record in history if 'validation_loss' in record),
               key=lambda record: record['validation_loss'])
    output_dir = Path('results') / 'camels' / basin / 'two_du_serial_seed42'
    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_checkpoint(
        output_dir / 'checkpoint',
        metadata={
            'basin': basin,
            'seed': 42,
            'configuration': 'two_du_serial',
            'training_epochs': len(history),
            'selected_epoch': best['epoch'],
            'selection': 'minimum validation MSE',
            'train_windows': list(train.water.shape),
            'validation_window': list(validation.water.shape),
        },
    )
    (output_dir / 'history.json').write_text(
        json.dumps(history, indent=2), encoding='utf-8'
    )
    print('Selected epoch:', best['epoch'])
    print('Checkpoint:', output_dir / 'checkpoint', flush=True)

print('Validation:', model.evaluate(validation)['metrics']['streamflow'])
if mode == 'full':
    print('Test:', model.evaluate(test)['metrics']['streamflow'])
else:
    print('The test period is not evaluated in check or benchmark mode.')
    print('These runs measure numerical execution and timing, not final predictive performance.')
